# E15-B — training `e15_new`: `e12_final`'s recipe, with new benign query rows injected

Registration: `research/e15/PREREG_E15.md` §§10, 12, 13. Everything is `e12_final`'s:
the same start, data, seed, batches, 20,000 steps, objective and pinned loss constant.
**One change:** 57,292 new same-population benign URLs with queries are added alongside
the old rows. Whenever a batch holds an old benign domain-frame row with a query, one
new row of the same kind joins the batch. The new rows teach **only their query**;
everything else in them is input. The old rows train exactly as before.

**Before any training, cell 3 rebuilds E11's start and checks it**, and checks that a
slightly larger batch at 384 tokens fits. **Cell 5 reads the run back:** it must print
`MISMATCHES: none -- e15_new is as registered`, including the old rows' batch digest
equal to the laptop's replay.

**Attach two inputs** (right panel → *Add Input*):
1. **`uie-corpus2`**, the data bundle E7–E14 used;
2. **`uie-e15-code`**.

**Settings:** GPU T4, Internet **On**. **Save Version → Save & Run All.** About 1.5 h.
**A rerun skips a finished arm.** When done, download `/kaggle/working`.

In [ ]:
# --- 1. locate the code bundle -----------------------------------------------------
import hashlib, json
from pathlib import Path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


root = Path("/kaggle/input")
code_hits = sorted(root.rglob("E15_CODE_MANIFEST.json")) if root.exists() else []
if not code_hits:
    raise SystemExit("Attach the uie-e15-code dataset (right panel -> Add Input), then rerun.")
CODE = code_hits[0].parent
WORK = Path("/kaggle/working")
print("code:", CODE)

data_hits = sorted(root.rglob("bundle.manifest.json")) if root.exists() else []
if not data_hits:
    raise SystemExit("Attach the uie-corpus2 dataset too, then rerun.")
DATA = data_hits[0].parent
print("data:", DATA)

In [ ]:
# --- 2. environment: install first, then run THIS code and prove it ------------
import importlib, subprocess, sys
from importlib import metadata

TLDEXTRACT = "5.3.1"          # the version the local evaluation parses with


def installed(dist):
    try:
        return metadata.version(dist)
    except metadata.PackageNotFoundError:
        return None


if installed("tldextract") != TLDEXTRACT:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        f"tldextract=={TLDEXTRACT}"], check=True)
    except subprocess.CalledProcessError:
        raise SystemExit("pip could not install tldextract: Settings -> Internet -> On, rerun.")
    importlib.invalidate_caches()
assert installed("tldextract") == TLDEXTRACT, installed("tldextract")

# CODE ONLY on the path: the data bundle carries an older `uie`.
for p in (CODE / "research" / "e15", CODE / "research" / "e13", CODE / "research" / "native", CODE):
    sys.path.insert(0, str(p))
for name in [m for m in sys.modules if m == "uie" or m.startswith("uie.")
             or m in ("e8_common", "e11_common", "e12_common", "e13_common", "e15_common")]:
    del sys.modules[name]
import uie
import e12_common as K
import e15_common as E15
manifest = json.loads((CODE / "E15_CODE_MANIFEST.json").read_text())
for mod in (uie, K, E15):
    assert str(CODE) in mod.__file__, f"{mod.__name__} is SHADOWED -- check sys.path"
bad = [f for f, h in manifest["files"].items() if sha256(CODE / f) != h]
bad += [t for t, h in manifest["tables"].items() if sha256(CODE / t) != h]
if sha256(CODE / "e12_final.pt") != manifest["e12_final"]:
    bad.append("e12_final.pt")
if sha256(CODE / E15.STREAM_FILE) != manifest["stream"]["sha256"]:
    bad.append(E15.STREAM_FILE)
assert not bad, f"STOP. files differ from the manifest: {bad}"
print(f"code: commit {manifest['commit'][:12]}; {len(manifest['files'])} files, "
      f"{len(manifest['tables'])} tables, e12_final.pt and the stream ({manifest['stream']['rows']:,} rows) verified")

import torch
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Settings -> Accelerator -> GPU T4, then rerun.")
GPU = torch.cuda.get_device_name(0)
print("torch:", torch.__version__, "| gpu:", GPU)

need = ["name_train", "structure_train", "phishing_name_train", "phishing_structure_train",
        "name_val", "structure_val", "phishing_name_val", "phishing_structure_val"]
present = {p.name for p in DATA.iterdir()}
missing = [s for s in need if f"{s}.jsonl" not in present and f"{s}.jsonl.gz" not in present]
if missing:
    raise SystemExit(f"STOP. missing {missing}: this is not the uie-corpus2 bundle.")
import inspect
from uie.kaggle_train import run
assert {"qry_extra", "slash_plan", "max_len", "e12_start", "loss_constant"} <= set(
    inspect.signature(run).parameters)
print("data bundle: all 8 streams present; E15 option present")
RECIPE = E15.recipe_new(CODE / E15.STREAM_FILE)
print("e15_new:", RECIPE)

In [ ]:
# --- 3. THE START CHECK and the memory preflight, before any training -----------------
e10, e11 = K.start_fingerprints()
print("E10 start:", e10, "(recorded", K.E10_START + ")")
print("E11 start:", e11, "(recorded", K.E11_START + ")")
if (e10, e11) != (K.E10_START, K.E11_START):
    raise SystemExit("STOP. This torch rebuilds a different start than E10/E11 recorded. "
                     "Nothing was trained.")
try:
    # 256 rows plus a margin for the appended new rows (about 4.7 per batch on average)
    peak = K.memory_preflight(torch.device("cuda"), batch_size=272)
except torch.cuda.OutOfMemoryError:
    raise SystemExit("STOP. A 272 x 384 training step does not fit on this GPU.")
total = torch.cuda.get_device_properties(0).total_memory / 2**30
print(f"start check PASSED; memory preflight PASSED ({peak:.1f} of {total:.1f} GB)")

In [ ]:
# --- 4. train e15_new (skipped if finished) -----------------------------------------
import time

PROV = WORK / "PROVENANCE.json"
prov = json.loads(PROV.read_text()) if PROV.exists() else {"arms": {}}
prov.update({"registration": E15.REGISTRATION, "code_commit": manifest["commit"],
             "start_check": {"e10": e10, "e11": e11}, "gpu": GPU,
             "stream_sha256": manifest["stream"]["sha256"]})
out = WORK / E15.ARM
if (out / "train_result.json").exists() and (out / "final.pt").exists():
    print(f"{E15.ARM}: finished earlier -- skipped")
else:
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    run(DATA, out, **RECIPE)
    prov["arms"][E15.ARM] = {"wall_minutes": round((time.time() - t0) / 60, 1)}
    PROV.write_text(json.dumps(prov, indent=1))

In [ ]:
# --- 5. read e15_new back against e12_final's record and Amendment 3 ------------------
ck = torch.load(WORK / E15.ARM / "final.pt", map_location="cpu", weights_only=False)
e12 = torch.load(CODE / "e12_final.pt", map_location="cpu", weights_only=False)
tc, mc, tk = ck["train_config"], ck["model_config"], ck["tokenizer"]
r = json.loads((WORK / E15.ARM / "train_result.json").read_text())
s = r.get("e12_start") or {}
q = r.get("qry_extra") or {}
strata = q.get("strata", {})
want = [(f"train_config {k}", tc.get(k), v) for k, v in e12["train_config"].items()]
want += [(f"model_config {k}", mc.get(k), v) for k, v in e12["model_config"].items()]
want += [("extra settings", sorted(set(tc) - set(e12["train_config"])),
          ["qry_extra", "qry_extra_draw_seed", "qry_extra_rate_seed"]),
         ("draw / rate seeds", (tc.get("qry_extra_draw_seed"), tc.get("qry_extra_rate_seed")),
          (E15.DRAW_SEED, E15.RATE_SEED)),
         ("tokenizer", tk == e12["tokenizer"], True),
         ("init weights", r.get("init_weights_sha256"), E15.E12_FINAL_INIT),
         ("row counts", r.get("counts"), E15.E12_FINAL_COUNTS),
         ("dropped over max_len", r.get("dropped_over_max_len"), E15.E12_FINAL_DROPPED),
         ("loss constant", r.get("loss_constant"), E15.E12_FINAL_LOSS_CONSTANT),
         ("stream sha256", q.get("source_sha256"), manifest["stream"]["sha256"]),
         ("N_new", {k: v["N_new"] for k, v in strata.items()}, E15.N_NEW),
         ("N_old", {k: v["N_old"] for k, v in strata.items()}, E15.N_OLD),
         ("p", {k: v["p"] for k, v in strata.items()}, E15.P_INJECT),
         ("old rows' batch digest (V-batch)", q.get("old_batch_digest"), E15.V_BATCH_DIGEST)]
want += [(f"start: {k}", s.get(k), v) for k, v in E15.E12_FINAL_START.items()]
problems = [w for w in want if w[1] != w[2]]
appended = sum(v["appended_rows"] for v in strata.values())
positions = sum(v["appended_query_positions"] for v in strata.values())
prov["arms"].setdefault(E15.ARM, {})
prov["arms"][E15.ARM].update({"best_sha256": sha256(WORK / E15.ARM / "best.pt"),
                              "final_sha256": sha256(WORK / E15.ARM / "final.pt"),
                              "init_weights_sha256": r.get("init_weights_sha256"),
                              "qry_extra": q, "final_val": r.get("final"), "counts": r.get("counts")})
PROV.write_text(json.dumps(prov, indent=1))
print(f"{E15.ARM} val {r.get('final')}")
print(f"injected: {appended:,} rows ({appended / max(q.get('steps') or 1, 1):.3f} per batch; "
      f"registered {E15.APPENDED_PER_BATCH}), {positions:,} query positions")
print("MISMATCHES:", problems if problems else f"none -- {E15.ARM} is as registered")
print("\nNow download /kaggle/working (e15_new/, PROVENANCE.json).")